# RLS Sales Cloud

Access conditioned on record type — the same dimension can mean something different depending on what kind of record is being viewed.

- **Universal roles** (`Country`, `Brand`) — same restriction across all record types, standard roles on their dimension tables.
- **Dataframe prep** — a per-user flag (`Is_Unrestricted_*`), aggregated before any row duplication, plus a row explosion that pairs each conditional dimension with the record type it applies to.
- **Concatenated roles** — `CompanyCode` (Orders only), `IndustryCode` and `CompanySolution` (Opportunities/Pipeline), combining the value with its record type via `concatenated_from` + `is_numeric`.
- **Fallback roles** — `Unrestricted_OppsPipeline`/`Unrestricted_Offers` grant full access on those record types when the precomputed flag says the user has no restriction, even if that same user is restricted on Orders (which has no fallback).

Full write-up: [`docs/CONFIGURATION_REFERENCE.md`](../docs/CONFIGURATION_REFERENCE.md).

### Setup

In [ ]:
workspace = 'DEV_YourWorkspace_Sales'

In [ ]:
dataset = "SALES CLOUD"

In [ ]:
%run Connections

In [ ]:
%run Email notifications

In [ ]:
%run RLS_Management_Functions

In [ ]:
workspace_current = workspace

### Global filters

In [ ]:
global_filters = []  # Sales Cloud no tiene concepto de Is_Consolidated

### Config

### RLS Table

Carga la tabla origen y calcula los flags de fallback (`Is_Unrestricted_*`).

In [ ]:
source_path = path_silver
source_table = 'rls_sf_sales'

df = spark.read.format("delta").load(f"{source_path}{source_table}")

# ── Fallback: "sin restricción en ninguna dimensión de Opportunity → ve todo" ──
# Replica la lógica del Case 2/3 del DAX dinámico original: si el usuario no
# tiene NINGÚN valor en Brand, Country, Industry ni CompanySolution (en
# NINGUNA de sus filas), ve todas las Opportunities/Pipeline. Los blancos en
# esta tabla vienen como "" además de NULL, así que hay que cubrir ambos.
# Para Offers (Case 4), el fallback solo mira Brand y Country — Industry y
# CompanySolution ni se mencionan en esa rama del DAX original.
def _has_value(col_name):
    return (
        F.col(col_name).isNotNull()
        & (F.col(col_name) != "")
        & (F.col(col_name) != "None")
    )

user_flags = (
    df.groupBy("Username")
    .agg(
        F.max(F.when(_has_value("Brand"), 1).otherwise(0)).alias("has_brand"),
        F.max(F.when(_has_value("CountryCode"), 1).otherwise(0)).alias("has_country"),
        F.max(F.when(_has_value("IndustryCode"), 1).otherwise(0)).alias("has_industry"),
        F.max(F.when(_has_value("CompanySolution"), 1).otherwise(0)).alias("has_solution"),
    )
    .withColumn(
        "Is_Unrestricted_478",
        F.when(
            (F.col("has_brand") == 0) & (F.col("has_country") == 0)
            & (F.col("has_industry") == 0) & (F.col("has_solution") == 0),
            F.lit("1")
        ).otherwise(F.lit("0"))
    )
    .withColumn(
        "Is_Unrestricted_6",
        F.when(
            (F.col("has_brand") == 0) & (F.col("has_country") == 0),
            F.lit("1")
        ).otherwise(F.lit("0"))
    )
    .select("Username", "Is_Unrestricted_478", "Is_Unrestricted_6")
)

df = df.join(user_flags, on="Username", how="left")

# CompanyCode aplica solo a Orders (DocType 7) — basta con etiquetar las filas,
# sin duplicar, porque solo hay un DocType posible.
df = df.withColumn(
    "DocType_CompanyCode",
    F.when(F.col("CompanyCode").isNotNull(), F.lit("7"))
)

# IndustryCode aplica a Opportunities y Pipeline (DocType 4 y 8) — SÍ hay que
# duplicar cada fila con IndustryCode poblado, una copia por DocType.
df_industry_base = df.where(F.col("IndustryCode").isNotNull())
df_industry_4 = df_industry_base.withColumn("DocType_Industry", F.lit("4"))
df_industry_8 = df_industry_base.withColumn("DocType_Industry", F.lit("8"))

# Mismo patrón para CompanySolution (también DocType 4 y 8)
df_solution_base = df.where(F.col("CompanySolution").isNotNull())
df_solution_4 = df_solution_base.withColumn("DocType_Solution", F.lit("4"))
df_solution_8 = df_solution_base.withColumn("DocType_Solution", F.lit("8"))

# Unir todo en un único dataframe. unionByName con allowMissingColumns rellena
# con null las columnas que no apliquen a cada fragmento.
df = (
    df
    .unionByName(df_industry_4, allowMissingColumns=True)
    .unionByName(df_industry_8, allowMissingColumns=True)
    .unionByName(df_solution_4, allowMissingColumns=True)
    .unionByName(df_solution_8, allowMissingColumns=True)
)

In [ ]:
config = {
    # ── UNIVERSALES (sin condición de DocType) ──────────────────────────────
    "Country": {
        "table": "DT_Country", "prefix": "RLS_Country",
        "column": "CountryCode", "source_column": "CountryCode"
    },
    "Brand": {
        "table": "DT_Brand", "prefix": "RLS_Brand",
        "column": "Brand", "source_column": "Brand"
    },

    # ── CONDICIONADAS POR DOCTYPE, vía roles concatenados ───────────────────
    "CompanyCode_Orders": {
        "prefix": "RLS_DocType7_CompanyCode",
        "concatenated_from": {
            "CompanyCode":         {"table": "FT_Sales", "column": "CompanyCode"},
            "DocType_CompanyCode": {"table": "FT_Sales", "column": "DocType", "is_numeric": True}
        }
    },
    "IndustryCode_OppsPipeline": {
        "prefix": "RLS_DocType478_IndustryCode",
        "concatenated_from": {
            "IndustryCode":     {"table": "FT_Sales", "column": "IndustryCode"},
            "DocType_Industry": {"table": "FT_Sales", "column": "DocType", "is_numeric": True}
        }
    },
    "CompanySolution_OppsPipeline": {
        "prefix": "RLS_DocType478_CompanySolution",
        "concatenated_from": {
            "CompanySolution":  {"table": "DT_Solution", "column": "Solution"},
            "DocType_Solution": {"table": "FT_Sales",     "column": "DocType", "is_numeric": True}
        }
    },

    # ── FALLBACK: "sin restricción en Opportunity/Offers → ve todo" ─────────
    "Unrestricted_OppsPipeline": {
        "table": "FT_Sales",
        "prefix": "RLS_DocType478_Unrestricted",
        "special": "consolidated",
        "fixed_filter": "[DocType] IN {4, 8}",
        "source_column": "Is_Unrestricted_478",
        "source_value": "1",
        "ignore_cols": [c for c in df.columns if c != "Is_Unrestricted_478"]
    },
    "Unrestricted_Offers": {
        "table": "FT_Sales",
        "prefix": "RLS_DocType6_Unrestricted",
        "special": "consolidated",
        "fixed_filter": "[DocType] = 6",
        "source_column": "Is_Unrestricted_6",
        "source_value": "1",
        "ignore_cols": [c for c in df.columns if c != "Is_Unrestricted_6"]
    },

    # "FullAccess": {
    #     "table": None, "prefix": "RLS_FullAccess", "column": None,
    #     "special": "full_access", "source_column": "FullAccess", "source_value": "ALL"
    # },
}

### AD Users join (validación de UPNs)

In [ ]:
source_path = path_bronze
source_table = 'ad_userslicences'

df_ad = spark.read.format("delta").load(f"{source_path}{source_table}").select(
    F.col('UserPrincipalName').alias('Username'),
    F.col('UserId').alias('User_ID')
)

#### USER VALIDATION
df = df.join(df_ad, on = 'Username', how='inner')
df = df.where(F.col('User_ID').isNotNull())


## RUN

In [ ]:
df, config = preprocess_concatenated_roles(df, config)

In [ ]:
config["Unrestricted_OppsPipeline"]["ignore_cols"] = list(df.columns)
config["Unrestricted_Offers"]["ignore_cols"] = list(df.columns)

#### 1. Crear/actualizar roles

In [ ]:
create_or_replace_roles_v3(config, dataset, workspace, global_filters, rls=df)

#### 2. Limpiar roles no usados

In [ ]:
#drop_unused_roles(config, dataset, workspace, rls=df)

#### 3. Sincronizar membership (delta, con auditoría)

In [ ]:
result, summary = run_with_audit(
    update_members_delta,
    dataset=dataset,
    config=config,      
    workspace=workspace,
    env=env,
    rls=df,
    chunk_size=1000,             
    alert_threshold=0.05,
    critical_drop_threshold=0.5,
    audit_path=path_control,     
    alert_to="your-team@yourcompany.com",
    skip_members=[]
)


In [ ]:
# # Renombra columnas en las tablas ya existentes, una sola vez
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN entorno TO environment")
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN modelo TO model")
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN miembros_antes TO members_before")
# spark.sql("ALTER TABLE rls_run_summary RENAME COLUMN miembros_despues TO members_after")
# # (y lo mismo para rls_change_log: entorno → environment, modelo → model)